In [2]:
import openmeteo_requests

import requests_cache
import pandas as pd
from retry_requests import retry

# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = -1)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
url = "https://archive-api.open-meteo.com/v1/archive"
params = {
	"latitude": 35.7778,
    "longitude": 10.8319,
	"start_date": "2000-01-01",
	"end_date": "2010-01-01",
	"daily": ["cloud_cover_max", "cloud_cover_min", "dew_point_2m_max", "dew_point_2m_min", "relative_humidity_2m_max", "relative_humidity_2m_min", "wind_gusts_10m_min", "wind_speed_10m_min"],
	"timezone": "auto"
}
responses = openmeteo.weather_api(url, params=params)

# Process first location. Add a for-loop for multiple locations or weather models
response = responses[0]
print(f"Coordinates {response.Latitude()}°N {response.Longitude()}°E")
print(f"Elevation {response.Elevation()} m asl")
print(f"Timezone {response.Timezone()}{response.TimezoneAbbreviation()}")
print(f"Timezone difference to GMT+0 {response.UtcOffsetSeconds()} s")

# Process daily data. The order of variables needs to be the same as requested.

daily = response.Daily()
daily_cloud_cover_max = daily.Variables(1).ValuesAsNumpy()
daily_cloud_cover_min = daily.Variables(2).ValuesAsNumpy()
daily_dew_point_2m_max = daily.Variables(3).ValuesAsNumpy()
daily_dew_point_2m_min = daily.Variables(4).ValuesAsNumpy()
daily_relative_humidity_2m_max = daily.Variables(5).ValuesAsNumpy()
daily_relative_humidity_2m_min = daily.Variables(6).ValuesAsNumpy()
daily_wind_gusts_10m_min = daily.Variables(7).ValuesAsNumpy()
daily_wind_speed_10m_min = daily.Variables(8).ValuesAsNumpy()

daily_data = {"date": pd.date_range(
	start = pd.to_datetime(daily.Time(), unit = "s", utc = True),
	end = pd.to_datetime(daily.TimeEnd(), unit = "s", utc = True),
	freq = pd.Timedelta(seconds = daily.Interval()),
	inclusive = "left"
)}

daily_data["cloud_cover_max"] = daily_cloud_cover_max
daily_data["cloud_cover_min"] = daily_cloud_cover_min
daily_data["dew_point_2m_max"] = daily_dew_point_2m_max
daily_data["dew_point_2m_min"] = daily_dew_point_2m_min
daily_data["relative_humidity_2m_max"] = daily_relative_humidity_2m_max
daily_data["relative_humidity_2m_min"] = daily_relative_humidity_2m_min
daily_data["wind_gusts_10m_min"] = daily_wind_gusts_10m_min
daily_data["wind_speed_10m_min"] = daily_wind_speed_10m_min

daily_dataframe = pd.DataFrame(data = daily_data)
print(daily_dataframe)

# Save the data as CSV and JSON files
daily_dataframe.to_csv('daily_weather_data_open_meteo_Monastir_f.csv', index=False)
daily_dataframe.to_json('daily_weather_data_open_meteo_Monastir_f.json', orient='records', lines=True)

print("Data saved as CSV and JSON files.")
print(daily_dataframe)


Coordinates 35.74692153930664°N 10.78608226776123°E
Elevation 5.0 m asl
Timezone b'Africa/Tunis'b'GMT+1'
Timezone difference to GMT+0 3600 s


error: unpack_from requires a buffer of at least 4294894310 bytes for unpacking 4 bytes at offset 4294894306 (actual buffer size is 117312)

In [17]:
import pandas as pd

df = pd.read_csv("dataset_part2.csv")
df.head()


,date,cloud_cover_max (%),cloud_cover_min (%),dew_point_2m_max (°C),dew_point_2m_min (°C),relative_humidity_2m_max (%),relative_humidity_2m_min (%),wind_gusts_10m_min (km/h),wind_speed_10m_min (km/h)
0,2000-01-01,89,2,5.8,2.6,86,60,16.2,8.6
1,2000-01-02,72,6,5.1,2.2,85,49,20.2,11.7
2,2000-01-03,47,24,7.0,3.4,81,63,19.4,10.7
3,2000-01-04,99,6,5.5,2.4,80,48,14.0,9.7
4,2000-01-05,62,13,7.3,3.4,90,58,8.6,5.7


In [19]:
df1 = pd.read_csv("daily_weather_data_cleaned.csv")
df1.head()

,date,weather_code,temperature_2m_max (°C),temperature_2m_min (°C),temperature_2m_mean (°C),daylight_duration (s),sunshine_duration (s),precipitation_sum (mm),rain_sum (mm),snowfall_sum (mm),precipitation_hours (hour),wind_speed_10m_max (km/h),wind_gusts_10m_max (km/h)
0,2000-01-01,51.0,12.841000,6.491000,9.497250,35240.280,29380.857,0.3,0.3,0.0,1.0,22.862125,39.239998
1,2000-01-02,53.0,12.540999,7.541000,9.730584,35278.860,31600.023,2.5,2.5,0.0,6.0,23.177402,44.639996
2,2000-01-03,3.0,13.891000,8.540999,10.686835,35320.746,31651.047,0.0,0.0,0.0,0.0,16.055353,29.880000
3,2000-01-04,2.0,14.240999,5.191000,9.376416,35365.832,31703.680,0.0,0.0,0.0,0.0,11.988594,24.840000
4,2000-01-05,2.0,14.341000,6.991000,10.193084,35414.023,31757.799,0.0,0.0,0.0,0.0,11.570515,21.240000


In [22]:
import pandas as pd

# Fusionner df et df2 sur la colonne 'date'
df_final = pd.merge(df, df1, on="date", how="inner")  # "inner" garde uniquement les dates communes

# Afficher les premières lignes du DataFrame fusionné
df_final.head()


,date,cloud_cover_max (%),cloud_cover_min (%),dew_point_2m_max (°C),dew_point_2m_min (°C),relative_humidity_2m_max (%),relative_humidity_2m_min (%),wind_gusts_10m_min (km/h),wind_speed_10m_min (km/h),weather_code,...,temperature_2m_min (°C),temperature_2m_mean (°C),daylight_duration (s),sunshine_duration (s),precipitation_sum (mm),rain_sum (mm),snowfall_sum (mm),precipitation_hours (hour),wind_speed_10m_max (km/h),wind_gusts_10m_max (km/h)
0,2000-01-01,89,2,5.8,2.6,86,60,16.2,8.6,51.0,...,6.491000,9.497250,35240.280,29380.857,0.3,0.3,0.0,1.0,22.862125,39.239998
1,2000-01-02,72,6,5.1,2.2,85,49,20.2,11.7,53.0,...,7.541000,9.730584,35278.860,31600.023,2.5,2.5,0.0,6.0,23.177402,44.639996
2,2000-01-03,47,24,7.0,3.4,81,63,19.4,10.7,3.0,...,8.540999,10.686835,35320.746,31651.047,0.0,0.0,0.0,0.0,16.055353,29.880000
3,2000-01-04,99,6,5.5,2.4,80,48,14.0,9.7,2.0,...,5.191000,9.376416,35365.832,31703.680,0.0,0.0,0.0,0.0,11.988594,24.840000
4,2000-01-05,62,13,7.3,3.4,90,58,8.6,5.7,2.0,...,6.991000,10.193084,35414.023,31757.799,0.0,0.0,0.0,0.0,11.570515,21.240000


In [29]:
df_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9191 entries, 0 to 9190
Data columns (total 21 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   date                          9191 non-null   object 
 1   cloud_cover_max (%)           9191 non-null   int64  
 2   cloud_cover_min (%)           9191 non-null   int64  
 3   dew_point_2m_max (°C)         9191 non-null   float64
 4   dew_point_2m_min (°C)         9191 non-null   float64
 5   relative_humidity_2m_max (%)  9191 non-null   int64  
 6   relative_humidity_2m_min (%)  9191 non-null   int64  
 7   wind_gusts_10m_min (km/h)     9191 non-null   float64
 8   wind_speed_10m_min (km/h)     9191 non-null   float64
 9   weather_code                  9191 non-null   float64
 10  temperature_2m_max  (°C)      9191 non-null   float64
 11  temperature_2m_min  (°C)      9191 non-null   float64
 12  temperature_2m_mean  (°C)     9191 non-null   float64
 13  day

In [30]:
df_final=df_final.drop(columns=["temperature_2m_mean  (°C)"])

In [31]:
df_final.head()

,date,cloud_cover_max (%),cloud_cover_min (%),dew_point_2m_max (°C),dew_point_2m_min (°C),relative_humidity_2m_max (%),relative_humidity_2m_min (%),wind_gusts_10m_min (km/h),wind_speed_10m_min (km/h),weather_code,temperature_2m_max (°C),temperature_2m_min (°C),daylight_duration (s),sunshine_duration (s),precipitation_sum (mm),rain_sum (mm),snowfall_sum (mm),precipitation_hours (hour),wind_speed_10m_max (km/h),wind_gusts_10m_max (km/h)
0,2000-01-01,89,2,5.8,2.6,86,60,16.2,8.6,51.0,12.841000,6.491000,35240.280,29380.857,0.3,0.3,0.0,1.0,22.862125,39.239998
1,2000-01-02,72,6,5.1,2.2,85,49,20.2,11.7,53.0,12.540999,7.541000,35278.860,31600.023,2.5,2.5,0.0,6.0,23.177402,44.639996
2,2000-01-03,47,24,7.0,3.4,81,63,19.4,10.7,3.0,13.891000,8.540999,35320.746,31651.047,0.0,0.0,0.0,0.0,16.055353,29.880000
3,2000-01-04,99,6,5.5,2.4,80,48,14.0,9.7,2.0,14.240999,5.191000,35365.832,31703.680,0.0,0.0,0.0,0.0,11.988594,24.840000
4,2000-01-05,62,13,7.3,3.4,90,58,8.6,5.7,2.0,14.341000,6.991000,35414.023,31757.799,0.0,0.0,0.0,0.0,11.570515,21.240000


In [33]:
df_final.to_csv("final_weather_data_monastir.csv", index=False)
